In [1]:
import os
import yaml
import polars as pl
import altair as alt

In [2]:
pl.Config(tbl_rows=100)
pl.Config(fmt_str_lengths=150)
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [3]:
df = pl.read_parquet(
    "data/segments.parquet"
).sort(
    ["id", "date"]
).with_columns(
    days_diff = pl.col("date").diff().dt.total_days().over("id"),
    effort_diff = pl.col("effort_count").diff().over("id")
).with_columns(
    avg_daily_efforts = pl.col("effort_diff") / pl.col("days_diff")
).with_columns(
    pl.when(pl.col("city").str.contains("Praha") | pl.col("city").str.contains("Prague")).then(pl.lit('Praha')).when(pl.col('city').str.contains('Brno')).then(pl.lit('Brno')).otherwise(pl.col('city')).alias('city'),
    pl.when(pl.col("effort_diff") < 0).then(pl.lit(0)).otherwise(pl.col("effort_diff")).alias("effort_diff")
).with_columns(
    pl.col('date').dt.ordinal_day().alias('day'),
    pl.col('date').dt.week().alias('week'),
    pl.col('date').dt.year().alias('year')
)

In [7]:
with open("data/behani_top_20.yaml", 'r', encoding='utf-8') as file:
    # safe_load automatically parses a YAML list into a Python list
    celorepublika = yaml.safe_load(file)

with open("data/kolo_top_20.yaml", 'r', encoding='utf-8') as file:
    # safe_load automatically parses a YAML list into a Python list
    celorepublika_kolo = yaml.safe_load(file)
    
with open("data/kolo_top_30.yaml", 'r', encoding='utf-8') as file:
    # safe_load automatically parses a YAML list into a Python list
    celorepublika_kolo_30 = yaml.safe_load(file)

with open("data/ovaly.yaml", 'r', encoding='utf-8') as file:
    # safe_load automatically parses a YAML list into a Python list
    ovaly = yaml.safe_load(file)

In [45]:
df.sample(2)

id,name,activity_type,distance,average_grade,elevation_high,elevation_low,total_elevation_gain,start_latlng,end_latlng,country,state,city,effort_count,athlete_count,date,start_to_finish_distance,days_diff,effort_diff,avg_daily_efforts,day,week,year
i64,str,str,f64,f64,f64,f64,f64,list[f64],list[f64],str,str,str,i64,i64,"datetime[μs, CET]",f64,i64,i64,f64,i16,i8,i32
22051560,"""Benedikt - Parkoviště""","""Run""",1995.5,0.0,321.9,298.9,35.7,"[50.493445, 13.670016]","[50.49344, 13.670002]","""Czechia""","""Ústecký kraj""","""Most""",20688,620,2025-12-28 23:21:54 CET,0.001136,0,9,inf,362,52,2025
36270077,"""Rovinka Beranov""","""Ride""",948.2,0.4,486.8,482.4,6.4,"[49.400318, 15.631629]","[49.39477, 15.639538]","""Czechia""","""Southeast""","""Malý Beranov""",9799,1383,2025-04-26 23:38:32 CEST,0.841521,1,14,14.0,116,17,2025


In [47]:
pracovni_nepracovni = pl.read_parquet("../kniha-rok/data/pracovni-polopracovni-nepracovni-dny.parquet")
pracovni_nepracovni.sample(2)

datum,den,vikend,pracovni-nepracovni,tyden
date,i32,bool,str,str
2024-07-23,2,false,"""pracovni""","""normalni"""
2026-01-14,3,false,"""pracovni""","""normalni"""


In [55]:
df = df.with_columns(
    pl.col('date').cast(str).str.slice(0,10).str.to_date()
).join(
    pracovni_nepracovni.rename({'datum':'date'}),
    on='date',
    how='left'
).filter(
    pl.col('tyden') == 'normalni'
)

In [9]:
orez_extremu = df.filter(
    pl.col('name').is_in(celorepublika)
).select(
    pl.col('effort_diff')
).quantile(
    0.99
).item(
)

do_grafu_celorepublika = df.filter(
    pl.col('name').is_in(celorepublika)
).filter(
    pl.col('year') >= 2025
).with_columns(
    pl.when(pl.col('effort_diff') >= orez_extremu).then(orez_extremu).otherwise(pl.col('effort_diff')).alias('effort_diff')
).group_by(
    pl.col('date').dt.weekday()
).agg(
    pl.col('effort_diff').sum()
).sort(
    by=['date']
)

do_grafu_celorepublika

date,effort_diff
i8,f64
1,10129.0
2,12230.0
3,11527.0
4,10218.0
5,8691.0
6,11464.0
7,12053.0


In [11]:
alt.Chart(
    do_grafu_celorepublika,
    width = 400
).mark_bar(
).encode(
    alt.X('date:N'),
    alt.Y('effort_diff:Q')
)

alt.Chart(...)

In [13]:
orez_extremu_kolo = df.filter(
    pl.col('name').is_in(celorepublika_kolo)
).select(
    pl.col('effort_diff')
).quantile(
    0.99
).item(
)

do_grafu_celorepublika_kolo = df.filter(
    pl.col('name').is_in(celorepublika_kolo)
).with_columns(
    pl.when(pl.col('effort_diff') >= orez_extremu_kolo).then(orez_extremu_kolo).otherwise(pl.col('effort_diff')).alias('effort_diff')
).group_by(
    pl.col('date').dt.weekday()
).agg(
    pl.col('effort_diff').sum()
).sort(
    by=['date']
)

In [15]:
alt.Chart(
    do_grafu_celorepublika_kolo,
    width = 400
).mark_bar(
).encode(
    alt.X('date:N'),
    alt.Y('effort_diff:Q')
)

alt.Chart(...)

In [17]:
do_grafu_ovaly = df.filter(
    pl.col('name').is_in(ovaly)
).filter(
    pl.col('year') >= 2025
).group_by(
    pl.col('date').dt.weekday()
).agg(
    pl.col('effort_diff').sum()
).sort(
    by=['date']
)

do_grafu_ovaly

date,effort_diff
i8,i64
1,18940
2,64427
3,50378
4,44775
5,22200
6,21896
7,16925


In [19]:
alt.Chart(
    do_grafu_ovaly,
    width = 400
).mark_bar(
).encode(
    alt.X('date:N'),
    alt.Y('effort_diff:Q')
)

alt.Chart(...)